# Building a Language Model from Scratch

This notebook demonstrates how to build the foundational components of a modern language model (LLM) from scratch, following GPT-style architecture principles.

## Table of Contents
1. [Dependencies Setup](#dependencies)
2. [Data Preparation](#data-prep)
3. [Tokenization](#tokenization)
4. [Dataset Implementation](#dataset)
5. [Embedding Layers](#embeddings)
6. [Model Architecture](#model)
7. [Training Setup](#training)
8. [Evaluation](#evaluation)
9. [Text Generation](#generation)

---

## 1. Dependencies Setup <a id="dependencies"></a>

First, let's install and import all necessary dependencies.

In [ ]:
# Install required packages
!pip install torch tiktoken matplotlib

# Import libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import tiktoken
import urllib.request
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import numpy as np
import os
from typing import Optional, Tuple

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Check if CUDA is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 2. Data Preparation <a id="data-prep"></a>

Download and prepare the text corpus for training.

In [ ]:
def download_text_data(url: str, filename: str) -> str:
    """Download text data if not already present."""
    if not os.path.exists(filename):
        print(f"Downloading {filename}...")
        urllib.request.urlretrieve(url, filename)
        print(f"Downloaded {filename}")
    else:
        print(f"{filename} already exists")
    
    with open(filename, 'r', encoding='utf-8') as f:
        text = f.read()
    
    return text

# Download the text corpus
url = "https://www.gutenberg.org/files/74/74-0.txt"  # The Adventures of Tom Sawyer
filename = "the-verdict.txt"

text_data = download_text_data(url, filename)

print(f"Text length: {len(text_data):,} characters")
print(f"First 500 characters:")
print(text_data[:500])

## 3. Tokenization <a id="tokenization"></a>

Implement tokenization using tiktoken's BPE tokenizer.

In [ ]:
class Tokenizer:
    """Wrapper for tiktoken tokenizer."""
    
    def __init__(self, model_name: str = "gpt2"):
        self.tokenizer = tiktoken.get_encoding(model_name)
        self.vocab_size = self.tokenizer.n_vocab
    
    def encode(self, text: str) -> list:
        """Encode text to token IDs."""
        return self.tokenizer.encode(text)
    
    def decode(self, token_ids: list) -> str:
        """Decode token IDs to text."""
        return self.tokenizer.decode(token_ids)

# Initialize tokenizer
tokenizer = Tokenizer()
print(f"Vocabulary size: {tokenizer.vocab_size:,}")

# Test tokenization
sample_text = "Hello, world! This is a test."
tokens = tokenizer.encode(sample_text)
decoded = tokenizer.decode(tokens)

print(f"Original: {sample_text}")
print(f"Tokens: {tokens}")
print(f"Decoded: {decoded}")

# Tokenize the entire dataset
all_tokens = tokenizer.encode(text_data)
print(f"Total tokens: {len(all_tokens):,}")

## 4. Dataset Implementation <a id="dataset"></a>

Create a custom dataset class for sliding window input-target pairs.

In [ ]:
class GPTDatasetV1(Dataset):
    """Custom dataset for GPT-style language modeling."""
    
    def __init__(self, tokens: list, context_length: int, stride: int = 1):
        self.tokens = tokens
        self.context_length = context_length
        self.stride = stride
        
        # Calculate number of samples
        self.num_samples = max(0, (len(tokens) - context_length) // stride + 1)
    
    def __len__(self):
        return self.num_samples
    
    def __getitem__(self, idx):
        start_idx = idx * self.stride
        end_idx = start_idx + self.context_length
        
        # Input sequence
        input_chunk = torch.tensor(self.tokens[start_idx:end_idx], dtype=torch.long)
        
        # Target sequence (shifted by 1)
        target_chunk = torch.tensor(self.tokens[start_idx + 1:end_idx + 1], dtype=torch.long)
        
        return input_chunk, target_chunk

# Configuration
CONTEXT_LENGTH = 256
BATCH_SIZE = 8
STRIDE = 128

# Split data into train/validation
split_idx = int(0.9 * len(all_tokens))
train_tokens = all_tokens[:split_idx]
val_tokens = all_tokens[split_idx:]

print(f"Train tokens: {len(train_tokens):,}")
print(f"Validation tokens: {len(val_tokens):,}")

# Create datasets
train_dataset = GPTDatasetV1(train_tokens, CONTEXT_LENGTH, STRIDE)
val_dataset = GPTDatasetV1(val_tokens, CONTEXT_LENGTH, STRIDE)

print(f"Train samples: {len(train_dataset):,}")
print(f"Validation samples: {len(val_dataset):,}")

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=True)

# Test the dataset
sample_input, sample_target = train_dataset[0]
print(f"\nSample shapes:")
print(f"Input: {sample_input.shape}")
print(f"Target: {sample_target.shape}")
print(f"\nSample content:")
print(f"Input: {tokenizer.decode(sample_input.tolist()[:50])}...")
print(f"Target: {tokenizer.decode(sample_target.tolist()[:50])}...")

## 5. Embedding Layers <a id="embeddings"></a>

Implement token and positional embeddings.

In [ ]:
class EmbeddingLayer(nn.Module):
    """Combined token and positional embeddings."""
    
    def __init__(self, vocab_size: int, embed_dim: int, context_length: int):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, embed_dim)
        self.positional_embedding = nn.Embedding(context_length, embed_dim)
        self.embed_dim = embed_dim
        self.context_length = context_length
    
    def forward(self, input_ids):
        batch_size, seq_len = input_ids.shape
        
        # Token embeddings
        token_embeds = self.token_embedding(input_ids)
        
        # Positional embeddings
        positions = torch.arange(seq_len, device=input_ids.device)
        pos_embeds = self.positional_embedding(positions)
        
        # Combine embeddings
        embeddings = token_embeds + pos_embeds
        
        return embeddings

# Test embedding layer
EMBED_DIM = 256
embedding_layer = EmbeddingLayer(tokenizer.vocab_size, EMBED_DIM, CONTEXT_LENGTH)

# Test with sample data
test_input = sample_input.unsqueeze(0)  # Add batch dimension
test_embeddings = embedding_layer(test_input)

print(f"Input shape: {test_input.shape}")
print(f"Embedding shape: {test_embeddings.shape}")
print(f"Expected shape: [batch_size={1}, seq_len={CONTEXT_LENGTH}, embed_dim={EMBED_DIM}]")

## 6. Model Architecture <a id="model"></a>

Implement a simple GPT-style transformer model.

In [ ]:
class MultiHeadAttention(nn.Module):
    """Multi-head self-attention mechanism."""
    
    def __init__(self, embed_dim: int, num_heads: int, dropout: float = 0.1):
        super().__init__()
        assert embed_dim % num_heads == 0
        
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        
        self.qkv_proj = nn.Linear(embed_dim, 3 * embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)
        
        # Causal mask
        self.register_buffer('causal_mask', torch.tril(torch.ones(1024, 1024)))
    
    def forward(self, x):
        batch_size, seq_len, embed_dim = x.shape
        
        # Generate Q, K, V
        qkv = self.qkv_proj(x)
        qkv = qkv.reshape(batch_size, seq_len, self.num_heads, 3 * self.head_dim)
        qkv = qkv.permute(0, 2, 1, 3)  # [batch, heads, seq, 3*head_dim]
        
        q, k, v = qkv.chunk(3, dim=-1)
        
        # Scaled dot-product attention
        scores = torch.matmul(q, k.transpose(-2, -1)) / (self.head_dim ** 0.5)
        
        # Apply causal mask
        mask = self.causal_mask[:seq_len, :seq_len]
        scores = scores.masked_fill(mask == 0, float('-inf'))
        
        # Apply softmax and dropout
        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)
        
        # Apply attention to values
        attn_output = torch.matmul(attn_weights, v)
        
        # Reshape and project
        attn_output = attn_output.permute(0, 2, 1, 3).contiguous()
        attn_output = attn_output.reshape(batch_size, seq_len, embed_dim)
        
        return self.out_proj(attn_output)


class FeedForward(nn.Module):
    """Position-wise feed-forward network."""
    
    def __init__(self, embed_dim: int, ff_dim: int, dropout: float = 0.1):
        super().__init__()
        self.linear1 = nn.Linear(embed_dim, ff_dim)
        self.linear2 = nn.Linear(ff_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        return self.linear2(self.dropout(F.gelu(self.linear1(x))))


class TransformerBlock(nn.Module):
    """Single transformer block."""
    
    def __init__(self, embed_dim: int, num_heads: int, ff_dim: int, dropout: float = 0.1):
        super().__init__()
        self.attention = MultiHeadAttention(embed_dim, num_heads, dropout)
        self.feed_forward = FeedForward(embed_dim, ff_dim, dropout)
        self.ln1 = nn.LayerNorm(embed_dim)
        self.ln2 = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        # Self-attention with residual connection
        attn_out = self.attention(self.ln1(x))
        x = x + self.dropout(attn_out)
        
        # Feed-forward with residual connection
        ff_out = self.feed_forward(self.ln2(x))
        x = x + self.dropout(ff_out)
        
        return x


class GPTModel(nn.Module):
    """Simple GPT-style language model."""
    
    def __init__(self, vocab_size: int, embed_dim: int, context_length: int, 
                 num_heads: int, num_layers: int, ff_dim: int, dropout: float = 0.1):
        super().__init__()
        self.embedding = EmbeddingLayer(vocab_size, embed_dim, context_length)
        
        self.transformer_blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, ff_dim, dropout)
            for _ in range(num_layers)
        ])
        
        self.ln_final = nn.LayerNorm(embed_dim)
        self.lm_head = nn.Linear(embed_dim, vocab_size, bias=False)
        
        # Initialize weights
        self.apply(self._init_weights)
    
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
    
    def forward(self, input_ids, targets=None):
        # Embeddings
        x = self.embedding(input_ids)
        
        # Transformer blocks
        for block in self.transformer_blocks:
            x = block(x)
        
        # Final layer norm and language modeling head
        x = self.ln_final(x)
        logits = self.lm_head(x)
        
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        
        return logits, loss

# Model configuration
model_config = {
    'vocab_size': tokenizer.vocab_size,
    'embed_dim': 256,
    'context_length': CONTEXT_LENGTH,
    'num_heads': 8,
    'num_layers': 6,
    'ff_dim': 1024,
    'dropout': 0.1
}

# Initialize model
model = GPTModel(**model_config)
model = model.to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

# Test forward pass
test_input = test_input.to(device)
test_target = sample_target.unsqueeze(0).to(device)

with torch.no_grad():
    logits, loss = model(test_input, test_target)
    print(f"\nTest forward pass:")
    print(f"Input shape: {test_input.shape}")
    print(f"Logits shape: {logits.shape}")
    print(f"Loss: {loss.item():.4f}")

## 7. Training Setup <a id="training"></a>

Implement training and evaluation functions.

In [ ]:
def train_epoch(model, train_loader, optimizer, device):
    """Train for one epoch."""
    model.train()
    total_loss = 0
    num_batches = 0
    
    for batch_idx, (inputs, targets) in enumerate(train_loader):
        inputs, targets = inputs.to(device), targets.to(device)
        
        optimizer.zero_grad()
        logits, loss = model(inputs, targets)
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        
        total_loss += loss.item()
        num_batches += 1
        
        if batch_idx % 100 == 0:
            print(f"Batch {batch_idx:4d}/{len(train_loader)}: Loss = {loss.item():.4f}")
    
    return total_loss / num_batches


def evaluate(model, val_loader, device):
    """Evaluate the model."""
    model.eval()
    total_loss = 0
    num_batches = 0
    
    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            logits, loss = model(inputs, targets)
            total_loss += loss.item()
            num_batches += 1
    
    return total_loss / num_batches


def plot_losses(epochs_seen, tokens_seen, train_losses, val_losses):
    """Plot training and validation losses."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    # Plot by epochs
    ax1.plot(epochs_seen, train_losses, label="Training loss", marker='o')
    ax1.plot(epochs_seen, val_losses, label="Validation loss", marker='s')
    ax1.set_xlabel("Epochs")
    ax1.set_ylabel("Loss")
    ax1.legend()
    ax1.grid(True)
    ax1.set_title("Loss vs Epochs")
    
    # Plot by tokens
    ax2.plot(tokens_seen, train_losses, label="Training loss", marker='o')
    ax2.plot(tokens_seen, val_losses, label="Validation loss", marker='s')
    ax2.set_xlabel("Tokens seen")
    ax2.set_ylabel("Loss")
    ax2.legend()
    ax2.grid(True)
    ax2.set_title("Loss vs Tokens Seen")
    
    plt.tight_layout()
    plt.show()


# Training configuration
LEARNING_RATE = 3e-4
NUM_EPOCHS = 10
WARMUP_STEPS = 1000

# Initialize optimizer and scheduler
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)

# Learning rate scheduler
def get_lr(step, warmup_steps, max_lr, min_lr=0.1):
    if step < warmup_steps:
        return max_lr * step / warmup_steps
    else:
        decay_ratio = (step - warmup_steps) / (NUM_EPOCHS * len(train_loader) - warmup_steps)
        return min_lr + (max_lr - min_lr) * 0.5 * (1 + np.cos(np.pi * decay_ratio))

scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer, 
    lr_lambda=lambda step: get_lr(step, WARMUP_STEPS, 1.0) / LEARNING_RATE
)

print("Training setup complete!")
print(f"Learning rate: {LEARNING_RATE}")
print(f"Number of epochs: {NUM_EPOCHS}")
print(f"Warmup steps: {WARMUP_STEPS}")

## 8. Training Loop <a id="training"></a>

Train the model and track progress.

In [ ]:
# Training tracking
train_losses = []
val_losses = []
epochs_seen = []
tokens_seen = []

print("Starting training...")
print(f"Training batches per epoch: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")

for epoch in range(NUM_EPOCHS):
    print(f"\n=== Epoch {epoch + 1}/{NUM_EPOCHS} ===")
    
    # Train
    train_loss = train_epoch(model, train_loader, optimizer, device)
    
    # Evaluate
    val_loss = evaluate(model, val_loader, device)
    
    # Update scheduler
    scheduler.step()
    
    # Track progress
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    epochs_seen.append(epoch + 1)
    tokens_seen.append((epoch + 1) * len(train_loader) * BATCH_SIZE * CONTEXT_LENGTH)
    
    # Print progress
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch {epoch + 1:2d}: Train Loss = {train_loss:.4f}, Val Loss = {val_loss:.4f}, LR = {current_lr:.6f}")
    
    # Plot losses every few epochs
    if (epoch + 1) % 5 == 0 or epoch == NUM_EPOCHS - 1:
        plot_losses(epochs_seen, tokens_seen, train_losses, val_losses)

print("\nTraining completed!")

## 9. Text Generation <a id="generation"></a>

Implement text generation functionality.

In [ ]:
def generate_text(model, tokenizer, prompt: str, max_new_tokens: int = 100, 
                 temperature: float = 1.0, top_k: Optional[int] = None, device='cpu'):
    """Generate text using the trained model."""
    model.eval()
    
    # Encode the prompt
    input_ids = torch.tensor(tokenizer.encode(prompt), dtype=torch.long).unsqueeze(0).to(device)
    
    generated_ids = input_ids.clone()
    
    with torch.no_grad():
        for _ in range(max_new_tokens):
            # Get the last context_length tokens
            if generated_ids.size(1) > CONTEXT_LENGTH:
                input_chunk = generated_ids[:, -CONTEXT_LENGTH:]
            else:
                input_chunk = generated_ids
            
            # Forward pass
            logits, _ = model(input_chunk)
            
            # Get logits for the last token
            logits = logits[:, -1, :] / temperature
            
            # Apply top-k filtering if specified
            if top_k is not None:
                top_k_logits, top_k_indices = torch.topk(logits, top_k)
                logits = torch.full_like(logits, float('-inf'))
                logits.scatter_(1, top_k_indices, top_k_logits)
            
            # Sample from the distribution
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            
            # Append to generated sequence
            generated_ids = torch.cat([generated_ids, next_token], dim=1)
    
    # Decode the generated text
    generated_text = tokenizer.decode(generated_ids[0].tolist())
    return generated_text


# Test text generation
test_prompts = [
    "Once upon a time",
    "The quick brown fox",
    "In a world where",
    "The secret to happiness is"
]

print("=== Text Generation Examples ===")
for i, prompt in enumerate(test_prompts):
    print(f"\n--- Example {i+1} ---")
    print(f"Prompt: '{prompt}'")
    
    generated = generate_text(
        model=model,
        tokenizer=tokenizer,
        prompt=prompt,
        max_new_tokens=50,
        temperature=0.8,
        top_k=50,
        device=device
    )
    
    print(f"Generated: {generated}")
    print("-" * 80)

## 10. Model Evaluation and Analysis

Analyze the trained model's performance and characteristics.

In [ ]:
# Final evaluation
final_train_loss = evaluate(model, train_loader, device)
final_val_loss = evaluate(model, val_loader, device)

print("=== Final Model Evaluation ===")
print(f"Final Training Loss: {final_train_loss:.4f}")
print(f"Final Validation Loss: {final_val_loss:.4f}")
print(f"Training/Validation Loss Ratio: {final_train_loss/final_val_loss:.3f}")

# Calculate perplexity
train_perplexity = torch.exp(torch.tensor(final_train_loss))
val_perplexity = torch.exp(torch.tensor(final_val_loss))

print(f"\nPerplexity:")
print(f"Training Perplexity: {train_perplexity:.2f}")
print(f"Validation Perplexity: {val_perplexity:.2f}")

# Plot final loss curves
plot_losses(epochs_seen, tokens_seen, train_losses, val_losses)

# Save the model
torch.save({
    'model_state_dict': model.state_dict(),
    'model_config': model_config,
    'train_losses': train_losses,
    'val_losses': val_losses,
    'final_train_loss': final_train_loss,
    'final_val_loss': final_val_loss
}, 'gpt_model_checkpoint.pt')

print("\nModel saved as 'gpt_model_checkpoint.pt'")

## Summary

This notebook demonstrated how to build a language model from scratch, covering:

1. **Data Preparation**: Downloaded and preprocessed text data
2. **Tokenization**: Implemented BPE tokenization using tiktoken
3. **Dataset**: Created a custom dataset with sliding window approach
4. **Embeddings**: Implemented token and positional embeddings
5. **Model Architecture**: Built a GPT-style transformer model
6. **Training**: Implemented training loop with proper optimization
7. **Generation**: Added text generation capabilities
8. **Evaluation**: Analyzed model performance and characteristics

### Key Takeaways:
- The model learns to generate coherent text through next-token prediction
- Proper tokenization and embedding are crucial for good performance
- Attention mechanisms allow the model to focus on relevant context
- Training requires careful hyperparameter tuning and monitoring

### Next Steps:
- Experiment with different model sizes and architectures
- Try different datasets and tokenization strategies
- Implement more advanced generation techniques (beam search, nucleus sampling)
- Add model evaluation metrics (BLEU, perplexity on different domains)
- Explore fine-tuning on specific tasks